In [ ]:
%pip install pandas numpy scikit-learn matplotlib

Note: you may need to restart the kernel to use updated packages.


Loads pandas library.

Pandas helps us work with tables (like Excel sheets).

✅ read_csv() loads your dataset

✅ df = DataFrame (table of data)

✅ head() shows first 5 rows

In [5]:
import pandas as pd

# load dataset
df = pd.read_csv("final_sensor_dataset.csv")

print(df.head())


             timestamp  PM2.5  PM10  TEMP   humidity  gasValue        AQI  \
0  2013-03-01 00:00:00    4.0   4.0  -0.7  24.008842     115.3  16.666667   
1  2013-03-01 01:00:00    8.0   8.0  -1.1  26.013678     115.3  33.333333   
2  2013-03-01 02:00:00    7.0   7.0  -1.1  26.013678     114.9  29.166667   
3  2013-03-01 03:00:00    6.0   6.0  -1.4  24.007753     116.0  25.000000   
4  2013-03-01 04:00:00    3.0   3.0  -2.0  24.876783     116.4  12.500000   

        CACI  
0  60.059977  
1  59.117950  
2  59.604061  
3  59.087210  
4  60.980058  



PART 3: Convert Timestamp to Date-Time
 - 👉 Why?

Your timestamp is text:

2013-03-01 10:00:00


AI cannot understand text time.

So we convert it into date-time format.


PART 4: Extract Time Features


👉 What this creates:
From:

2013-03-01 10:00
We get:

hour = 10
day = 1
weekday = 4

👉 Why important?

Pollution depends on time:

🚗 traffic hours

🏭 workday vs weekend

🌙 night pollution trapping

This helps AI learn patterns.





In [7]:
df['timestamp'] = pd.to_datetime(df['timestamp'])

df['hour'] = df['timestamp'].dt.hour
df['day'] = df['timestamp'].dt.day
# df['minute'] = df['timestamp'].dt.minute
df['weekday'] = df['timestamp'].dt.weekday


PART 5: Create Future Prediction Columns

👉 MOST IMPORTANT STEP

This tells AI:

➡ what happens in the next hour.

Example:

AQI    now	AQI_next

80	   95

95	   110

AI learns:

👉 how AQI changes.


In [3]:
df['AQI_next'] = df['AQI'].shift(-1)
df['CACI_next'] = df['CACI'].shift(-1)
# df['AQI_next'] = df['AQI'].shift(-60)
# df['CACI_next'] = df['CACI'].shift(-60)


PART 6: Remove Empty Row

In [4]:
df = df.dropna()


PART 7: Select Features (Inputs)

👉 These are inputs to AI:

Pollution + weather + time.

Think:

➡ causes of pollution.

👉 X = input data for AI

👉 y = what we want to predict

🎯 AQI after 1 hour

🎯 CACI after 1 hour

In [7]:
features = [
    'PM2.5',
    'PM10',
    'TEMP',
    'humidity',
    'gasValue',
    'hour',
    'weekday'
]

df['hour'] = df['timestamp'].dt.hour
df['weekday'] = df['timestamp'].dt.weekday

df['AQI_next'] = df['AQI'].shift(-1)
df['CACI_next'] = df['CACI'].shift(-1)

df = df.dropna()

X = df[features]

y_aqi = df['AQI_next']
y_caci = df['CACI_next']



PART 8: Split Data into Training & Testing
 - Loads tool to split data.

👉 What happens:

80% data → training

20% data → testing

👉 Why?

We test if AI learned correctly.

Think:

📚 study → training

📝 exam → testing



In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train_aqi, y_test_aqi = train_test_split(
    X, y_aqi, test_size=0.2, random_state=42
)

_, _, y_train_caci, y_test_caci = train_test_split(
    X, y_caci, test_size=0.2, random_state=42
)

PART 9: Train the Model

We import the AI model.

👉 Why Random Forest?

✔ accurate

✔ beginner friendly

✔ works well with pollution data

👉 What happens here:

AI studies patterns between:

inputs → next AQI

This is the learning step.

In [9]:
from sklearn.ensemble import RandomForestRegressor

# AQI model
aqi_model = RandomForestRegressor(n_estimators=200, random_state=42)
aqi_model.fit(X_train, y_train_aqi)

# CACI model
caci_model = RandomForestRegressor(n_estimators=200, random_state=42)
caci_model.fit(X_train, y_train_caci)

print("Training completed ✅")


Training completed ✅


PART 10: Check Accuracy
 - Tool to measure prediction error.
 - AI predicts values for test data.

 👉 What is error?

Difference between:

predicted vs actual.

Example:

Actual AQI = 150

Predicted AQI = 140

Error = 10

✔ smaller error = better model


In [10]:
from sklearn.metrics import mean_absolute_error

aqi_pred = aqi_model.predict(X_test)
caci_pred = caci_model.predict(X_test)

print("AQI Error:", mean_absolute_error(y_test_aqi, aqi_pred))
print("CACI Error:", mean_absolute_error(y_test_caci, caci_pred))


AQI Error: 13.926978306635439
CACI Error: 3.070144417654578


PART 11: Predict Future AQI
 - This selects the latest row (current pollution).
 - pred_aqi = aqi_model.predict(sample)

  pred_caci = caci_model.predict(sample) 
  
  AI predicts next hour pollution.


In [11]:
sample = X.iloc[-1:]   # latest row

pred_aqi = aqi_model.predict(sample)
pred_caci = caci_model.predict(sample)

print("Predicted AQI (1 hr):", pred_aqi[0])
print("Predicted CACI (1 hr):", pred_caci[0])


Predicted AQI (1 hr): 67.79594833969563
Predicted CACI (1 hr): 57.36650016554729


PART 12: Convert AQI to Status
 - Defines function.
 - Classifies pollution level.

In [12]:
def aqi_status(aqi):
    if aqi <= 50: return "Good"
    elif aqi <= 100: return "Satisfactory"
    elif aqi <= 200: return "Moderate"
    elif aqi <= 300: return "Poor"
    else: return "Severe"

print("Future AQI Status:", aqi_status(pred_aqi[0]))


Future AQI Status: Satisfactory


In [13]:
import joblib

joblib.dump(aqi_model, "aqi_model.pkl")
joblib.dump(caci_model, "caci_model.pkl")

print("Models saved ✅")


Models saved ✅


## PART 13: Better Training for Higher Accuracy

This section improves model performance using:
- time-series friendly split (no random leakage)
- lag features (previous AQI/CACI values)
- cyclical time encoding (hour/day patterns)
- hyperparameter tuning with cross-validation

In [17]:
import numpy as np

# Work on a copy so earlier cells stay unchanged
df2 = df.copy()

# Resolve expected columns case-insensitively (handles gasValue vs gasvalue, etc.)
column_lookup = {c.lower(): c for c in df2.columns}
required_columns = {
    'AQI': 'aqi',
    'CACI': 'caci',
    'PM2.5': 'pm2.5',
    'PM10': 'pm10',
    'TEMP': 'temp',
    'humidity': 'humidity',
    'gasValue': 'gasvalue'
}

missing = [name for name, key in required_columns.items() if key not in column_lookup]
if missing:
    raise KeyError(f"Missing required columns: {missing}. Available columns: {list(df2.columns)}")

# Create canonical columns so all later code uses one consistent naming scheme
for canonical_name, lookup_key in required_columns.items():
    actual_name = column_lookup[lookup_key]
    if actual_name != canonical_name:
        df2[canonical_name] = df2[actual_name]

# Add lag features: previous observations often improve next-step forecasting
for col in ['AQI', 'CACI', 'PM2.5', 'PM10', 'gasValue']:
    df2[f'{col}_lag1'] = df2[col].shift(1)
    df2[f'{col}_lag2'] = df2[col].shift(2)

# Cyclical encoding for hour/day patterns
if 'hour' not in df2.columns:
    df2['hour'] = df2['timestamp'].dt.hour
if 'day' not in df2.columns:
    df2['day'] = df2['timestamp'].dt.day
if 'weekday' not in df2.columns:
    df2['weekday'] = df2['timestamp'].dt.weekday

# Better target horizon (next 1 step) from current row
df2['AQI_next'] = df2['AQI'].shift(-1)
df2['CACI_next'] = df2['CACI'].shift(-1)

df2['hour_sin'] = np.sin(2 * np.pi * df2['hour'] / 24)
df2['hour_cos'] = np.cos(2 * np.pi * df2['hour'] / 24)
df2['weekday_sin'] = np.sin(2 * np.pi * df2['weekday'] / 7)
df2['weekday_cos'] = np.cos(2 * np.pi * df2['weekday'] / 7)

df2 = df2.dropna().reset_index(drop=True)

base_features = ['PM2.5', 'PM10', 'gasValue', 'TEMP', 'humidity']
lag_features = [
    'AQI_lag1', 'AQI_lag2', 'CACI_lag1', 'CACI_lag2',
    'PM2.5_lag1', 'PM2.5_lag2', 'PM10_lag1', 'PM10_lag2',
    'gasValue_lag1', 'gasValue_lag2'
 ]
time_features = ['hour_sin', 'hour_cos', 'weekday_sin', 'weekday_cos', 'day']

features_v2 = base_features + lag_features + time_features
X2 = df2[features_v2]
y2_aqi = df2['AQI_next']
y2_caci = df2['CACI_next']

print('Improved dataset shape:', X2.shape)

Improved dataset shape: (31873, 20)


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV

# Time-series split: train on past, test on future
split_idx = int(len(X2) * 0.8)
X_train2, X_test2 = X2.iloc[:split_idx], X2.iloc[split_idx:]
y_train2_aqi, y_test2_aqi = y2_aqi.iloc[:split_idx], y2_aqi.iloc[split_idx:]
y_train2_caci, y_test2_caci = y2_caci.iloc[:split_idx], y2_caci.iloc[split_idx:]

tscv = TimeSeriesSplit(n_splits=5)

param_dist = {
    'n_estimators': [200, 300, 500, 700],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None]
}

base_rf = RandomForestRegressor(random_state=42, n_jobs=-1)

search_aqi = RandomizedSearchCV(
    estimator=base_rf,
    param_distributions=param_dist,
    n_iter=20,
    cv=tscv,
    scoring='neg_mean_absolute_error',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

search_caci = RandomizedSearchCV(
    estimator=base_rf,
    param_distributions=param_dist,
    n_iter=20,
    cv=tscv,
    scoring='neg_mean_absolute_error',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

search_aqi.fit(X_train2, y_train2_aqi)
search_caci.fit(X_train2, y_train2_caci)

aqi_model_best = search_aqi.best_estimator_
caci_model_best = search_caci.best_estimator_

pred2_aqi = aqi_model_best.predict(X_test2)
pred2_caci = caci_model_best.predict(X_test2)

aqi_mae = mean_absolute_error(y_test2_aqi, pred2_aqi)
caci_mae = mean_absolute_error(y_test2_caci, pred2_caci)
aqi_rmse = root_mean_squared_error(y_test2_aqi, pred2_aqi)
caci_rmse = root_mean_squared_error(y_test2_caci, pred2_caci)

print('Best AQI params:', search_aqi.best_params_)
print('Best CACI params:', search_caci.best_params_)
print('Improved AQI MAE :', round(aqi_mae, 3))
print('Improved AQI RMSE:', round(aqi_rmse, 3))
print('Improved CACI MAE :', round(caci_mae, 3))
print('Improved CACI RMSE:', round(caci_rmse, 3))

Fitting 5 folds for each of 20 candidates, totalling 100 fits


In [19]:
# Use improved models as the default models for later prediction cells
aqi_model = aqi_model_best
caci_model = caci_model_best

import joblib
joblib.dump(aqi_model, 'aqi_model_1.pkl')
joblib.dump(caci_model, 'caci_model_1.pkl')

print('Improved models saved as aqi_model.pkl and caci_model.pkl')

Improved models saved as aqi_model.pkl and caci_model.pkl


## PART 14: Predict TEMP, Humidity, and PM2.5

This section uses the same improved feature set from Part 13 and predicts next-step values for:
- `TEMP_next`
- `humidity_next`
- `PM2.5_next`

In [21]:
# Create next-step targets
work_df = df2.copy()
work_df['TEMP_next'] = work_df['TEMP'].shift(-1)
work_df['humidity_next'] = work_df['humidity'].shift(-1)
work_df['PM2.5_next'] = work_df['PM2.5'].shift(-1)
work_df = work_df.dropna().reset_index(drop=True)

X_extra = work_df[features_v2]
y_temp = work_df['TEMP_next']
y_humidity = work_df['humidity_next']
y_pm25 = work_df['PM2.5_next']

split_idx_extra = int(len(X_extra) * 0.8)
X_train_e, X_test_e = X_extra.iloc[:split_idx_extra], X_extra.iloc[split_idx_extra:]
y_train_temp, y_test_temp = y_temp.iloc[:split_idx_extra], y_temp.iloc[split_idx_extra:]
y_train_hum, y_test_hum = y_humidity.iloc[:split_idx_extra], y_humidity.iloc[split_idx_extra:]
y_train_pm25, y_test_pm25 = y_pm25.iloc[:split_idx_extra], y_pm25.iloc[split_idx_extra:]

# Reuse tuned settings from AQI model for consistency and speed
temp_model = RandomForestRegressor(**search_aqi.best_params_, random_state=42, n_jobs=-1)
humidity_model = RandomForestRegressor(**search_aqi.best_params_, random_state=42, n_jobs=-1)
pm25_model = RandomForestRegressor(**search_aqi.best_params_, random_state=42, n_jobs=-1)

temp_model.fit(X_train_e, y_train_temp)
humidity_model.fit(X_train_e, y_train_hum)
pm25_model.fit(X_train_e, y_train_pm25)

pred_temp = temp_model.predict(X_test_e)
pred_hum = humidity_model.predict(X_test_e)
pred_pm25 = pm25_model.predict(X_test_e)

print('TEMP MAE :', round(mean_absolute_error(y_test_temp, pred_temp), 3))
print('TEMP RMSE:', round(root_mean_squared_error(y_test_temp, pred_temp), 3))
print('Humidity MAE :', round(mean_absolute_error(y_test_hum, pred_hum), 3))
print('Humidity RMSE:', round(root_mean_squared_error(y_test_hum, pred_hum), 3))
print('PM2.5 MAE :', round(mean_absolute_error(y_test_pm25, pred_pm25), 3))
print('PM2.5 RMSE:', round(root_mean_squared_error(y_test_pm25, pred_pm25), 3))

TEMP MAE : 0.671
TEMP RMSE: 0.945
Humidity MAE : 3.69
Humidity RMSE: 5.604
PM2.5 MAE : 10.308
PM2.5 RMSE: 19.091


In [22]:
# Predict next-step TEMP/Humidity/PM2.5 from the latest row
sample_extra = X_extra.iloc[-1:]

future_temp = temp_model.predict(sample_extra)[0]
future_humidity = humidity_model.predict(sample_extra)[0]
future_pm25 = pm25_model.predict(sample_extra)[0]

print('Predicted TEMP (next step):', round(future_temp, 2))
print('Predicted humidity (next step):', round(future_humidity, 2))
print('Predicted PM2.5 (next step):', round(future_pm25, 2))

joblib.dump(temp_model, 'temp_model.pkl')
joblib.dump(humidity_model, 'humidity_model.pkl')
joblib.dump(pm25_model, 'pm25_model.pkl')
print('Saved: temp_model.pkl, humidity_model.pkl, pm25_model.pkl')

Predicted TEMP (next step): 11.46
Predicted humidity (next step): 13.2
Predicted PM2.5 (next step): 15.45
Saved: temp_model.pkl, humidity_model.pkl, pm25_model.pkl


## PART 15: Deployment-Safe Models (No AQI/CACI Lag Dependency)

This section trains a second set of models for production use without `AQI_lag*` and `CACI_lag*`,
so realtime prediction does not depend on previously predicted AQI/CACI values.

In [9]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
import joblib

# Reuse engineered table from Part 13
deploy_df = df2.copy()

# Production-safe feature set: remove AQI/CACI lag features
deploy_features = [
    'PM2.5', 'PM10', 'gasValue', 'TEMP', 'humidity',
    'PM2.5_lag1', 'PM2.5_lag2',
    'PM10_lag1', 'PM10_lag2',
    'gasValue_lag1', 'gasValue_lag2',
    'hour_sin', 'hour_cos', 'weekday_sin', 'weekday_cos', 'day'
 ]

X_deploy = deploy_df[deploy_features]
y_deploy_aqi = deploy_df['AQI_next']
y_deploy_caci = deploy_df['CACI_next']

split_idx_deploy = int(len(X_deploy) * 0.8)
X_train_d, X_test_d = X_deploy.iloc[:split_idx_deploy], X_deploy.iloc[split_idx_deploy:]
y_train_d_aqi, y_test_d_aqi = y_deploy_aqi.iloc[:split_idx_deploy], y_deploy_aqi.iloc[split_idx_deploy:]
y_train_d_caci, y_test_d_caci = y_deploy_caci.iloc[:split_idx_deploy], y_deploy_caci.iloc[split_idx_deploy:]

best_params_for_deploy = search_aqi.best_params_ if 'search_aqi' in globals() else {
    'n_estimators': 300,
    'max_depth': 20,
    'min_samples_split': 2,
    'min_samples_leaf': 1,
    'max_features': 'sqrt'
}

aqi_model_live = RandomForestRegressor(**best_params_for_deploy, random_state=42, n_jobs=-1)
caci_model_live = RandomForestRegressor(**best_params_for_deploy, random_state=42, n_jobs=-1)

aqi_model_live.fit(X_train_d, y_train_d_aqi)
caci_model_live.fit(X_train_d, y_train_d_caci)

pred_d_aqi = aqi_model_live.predict(X_test_d)
pred_d_caci = caci_model_live.predict(X_test_d)

print('Live AQI MAE :', round(mean_absolute_error(y_test_d_aqi, pred_d_aqi), 3))
print('Live AQI RMSE:', round(root_mean_squared_error(y_test_d_aqi, pred_d_aqi), 3))
print('Live CACI MAE :', round(mean_absolute_error(y_test_d_caci, pred_d_caci), 3))
print('Live CACI RMSE:', round(root_mean_squared_error(y_test_d_caci, pred_d_caci), 3))

# Train deployment-safe TEMP/Humidity/PM2.5 models with same feature set
deploy_extra = deploy_df.copy()
deploy_extra['TEMP_next'] = deploy_extra['TEMP'].shift(-1)
deploy_extra['humidity_next'] = deploy_extra['humidity'].shift(-1)
deploy_extra['PM2.5_next'] = deploy_extra['PM2.5'].shift(-1)
deploy_extra = deploy_extra.dropna().reset_index(drop=True)

X_deploy_extra = deploy_extra[deploy_features]
y_deploy_temp = deploy_extra['TEMP_next']
y_deploy_humidity = deploy_extra['humidity_next']
y_deploy_pm25 = deploy_extra['PM2.5_next']

split_idx_deploy_extra = int(len(X_deploy_extra) * 0.8)
X_train_dx, X_test_dx = X_deploy_extra.iloc[:split_idx_deploy_extra], X_deploy_extra.iloc[split_idx_deploy_extra:]
y_train_temp_dx, y_test_temp_dx = y_deploy_temp.iloc[:split_idx_deploy_extra], y_deploy_temp.iloc[split_idx_deploy_extra:]
y_train_hum_dx, y_test_hum_dx = y_deploy_humidity.iloc[:split_idx_deploy_extra], y_deploy_humidity.iloc[split_idx_deploy_extra:]
y_train_pm_dx, y_test_pm_dx = y_deploy_pm25.iloc[:split_idx_deploy_extra], y_deploy_pm25.iloc[split_idx_deploy_extra:]

temp_model_live = RandomForestRegressor(**best_params_for_deploy, random_state=42, n_jobs=-1)
humidity_model_live = RandomForestRegressor(**best_params_for_deploy, random_state=42, n_jobs=-1)
pm25_model_live = RandomForestRegressor(**best_params_for_deploy, random_state=42, n_jobs=-1)

temp_model_live.fit(X_train_dx, y_train_temp_dx)
humidity_model_live.fit(X_train_dx, y_train_hum_dx)
pm25_model_live.fit(X_train_dx, y_train_pm_dx)

pred_temp_dx = temp_model_live.predict(X_test_dx)
pred_hum_dx = humidity_model_live.predict(X_test_dx)
pred_pm_dx = pm25_model_live.predict(X_test_dx)

print('Live TEMP MAE :', round(mean_absolute_error(y_test_temp_dx, pred_temp_dx), 3))
print('Live TEMP RMSE:', round(root_mean_squared_error(y_test_temp_dx, pred_temp_dx), 3))
print('Live HUM MAE  :', round(mean_absolute_error(y_test_hum_dx, pred_hum_dx), 3))
print('Live HUM RMSE :', round(root_mean_squared_error(y_test_hum_dx, pred_hum_dx), 3))
print('Live PM2.5 MAE :', round(mean_absolute_error(y_test_pm_dx, pred_pm_dx), 3))
print('Live PM2.5 RMSE:', round(root_mean_squared_error(y_test_pm_dx, pred_pm_dx), 3))

joblib.dump(aqi_model_live, 'aqi_model_live.pkl')
joblib.dump(caci_model_live, 'caci_model_live.pkl')
joblib.dump(temp_model_live, 'temp_model_live.pkl')
joblib.dump(humidity_model_live, 'humidity_model_live.pkl')
joblib.dump(pm25_model_live, 'pm25_model_live.pkl')

print('Saved deployment-safe models: aqi_model_live.pkl, caci_model_live.pkl, temp_model_live.pkl, humidity_model_live.pkl, pm25_model_live.pkl')

Live AQI MAE : 13.329
Live AQI RMSE: 22.316
Live CACI MAE : 3.412
Live CACI RMSE: 4.629
Live TEMP MAE : 1.047
Live TEMP RMSE: 1.406
Live HUM MAE  : 3.936
Live HUM RMSE : 5.76
Live PM2.5 MAE : 10.333
Live PM2.5 RMSE: 19.449
Saved deployment-safe models: aqi_model_live.pkl, caci_model_live.pkl, temp_model_live.pkl, humidity_model_live.pkl, pm25_model_live.pkl


## PART 17: India Local Adaptation (ThingSpeak History -> Retrain Live Models)

This block builds a local training table from your ThingSpeak history (2-minute data),
then retrains deployment-safe `*_live.pkl` models.

Note: AQI/CACI models are retrained only if ground-truth AQI/CACI labels are available in the input history file.

In [3]:
import pandas as pd
import numpy as np
import joblib
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
try:
    import xgboost as xgb
    HAS_XGBOOST = True
except ImportError:
    HAS_XGBOOST = False
    print("⚠️  XGBoost not installed. Install with: pip install xgboost")

# ===== CONFIG =====
USE_LOCAL_EXPORT = False
LOCAL_EXPORT_PATH = "thingspeak_india_history.csv"  # optional CSV export path

# If using direct API fetch for adaptation data
ADAPT_CHANNEL_ID = "3220962"
ADAPT_READ_API_KEY = "TF7VPOAMFV8XK33V"
ADAPT_RESULTS = 8000

# ThingSpeak -> canonical mapping
ADAPT_SOURCE_MAP = {
    "field1": "TEMP",
    "field2": "humidity",
    "field3": "PM2.5",
    "field4": "PM10",
    "field7": "gasValue",
}


def load_adaptation_raw():
    if USE_LOCAL_EXPORT:
        raw = pd.read_csv(LOCAL_EXPORT_PATH)
    else:
        url = (
            f"https://api.thingspeak.com/channels/{ADAPT_CHANNEL_ID}/feeds.csv"
            f"?api_key={ADAPT_READ_API_KEY}&results={ADAPT_RESULTS}"
        )
        raw = pd.read_csv(url)
    return raw


def to_canonical_frame(raw):
    cols = set(raw.columns)
    keep = ["created_at"] + [c for c in ADAPT_SOURCE_MAP.keys() if c in cols]
    if "timestamp" in cols and "created_at" not in cols:
        raw = raw.rename(columns={"timestamp": "created_at"})
        keep = ["created_at"] + [c for c in ADAPT_SOURCE_MAP.keys() if c in raw.columns]

    missing = [c for c in ADAPT_SOURCE_MAP.keys() if c not in raw.columns]
    if missing:
        raise ValueError(f"Missing sensor fields in adaptation data: {missing}")

    df = raw[keep].copy()
    df = df.rename(columns=ADAPT_SOURCE_MAP)
    df["timestamp"] = pd.to_datetime(df["created_at"], errors="coerce")

    for c in ["TEMP", "humidity", "PM2.5", "PM10", "gasValue"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # Optional labels if present
    if "AQI" in raw.columns:
        df["AQI"] = pd.to_numeric(raw["AQI"], errors="coerce")
    if "CACI" in raw.columns:
        df["CACI"] = pd.to_numeric(raw["CACI"], errors="coerce")

    df = df.dropna(subset=["timestamp", "TEMP", "humidity", "PM2.5", "PM10", "gasValue"])
    df = df.sort_values("timestamp").reset_index(drop=True)
    return df


def build_engineered_from_india(df):
    """Enhanced feature engineering specifically optimized for PM2.5 prediction."""
    d = df.copy()

    # === MULTI-SCALE PM2.5 LAG FEATURES (15, 30, 45, 60-min windows) ===
    # 2-minute data: 7rows≈15min, 15rows≈30min, 22rows≈45min, 30rows≈60min
    window_configs = [
        (7, 3),    # 15-minute rolling window, shifted by 3 (6 minutes back)
        (15, 7),   # 30-minute rolling window, shifted by 7 (14 minutes back)
        (22, 11),  # 45-minute rolling window, shifted by 11 (22 minutes back)
        (30, 15),  # 60-minute rolling window, shifted by 15 (30 minutes back)
    ]
    
    for window, shift in window_configs:
        d[f"PM2.5_lag_w{window}"] = d["PM2.5"].rolling(window).mean().shift(shift)
        d[f"PM2.5_std_w{window}"] = d["PM2.5"].rolling(window).std().shift(shift)  # Volatility
    
    # === PM2.5 RATE OF CHANGE (trend indicator) ===
    d["PM2.5_roc_5min"] = d["PM2.5"].diff(2)  # ~5-minute change
    d["PM2.5_roc_10min"] = d["PM2.5"].diff(5)  # ~10-minute change
    d["PM2.5_roc_20min"] = d["PM2.5"].diff(10)  # ~20-minute change
    
    # === EXPONENTIAL MOVING AVERAGES for PM2.5 ===
    d["PM2.5_ema_7"] = d["PM2.5"].ewm(span=7).mean()
    d["PM2.5_ema_15"] = d["PM2.5"].ewm(span=15).mean()
    
    # === PM2.5 PERCENTILE FEATURES (robust to outliers) ===
    d["PM2.5_pct25_20min"] = d["PM2.5"].rolling(10).quantile(0.25)
    d["PM2.5_pct75_20min"] = d["PM2.5"].rolling(10).quantile(0.75)
    d["PM2.5_iqr_20min"] = d["PM2.5_pct75_20min"] - d["PM2.5_pct25_20min"]

    # === INTERACTION FEATURES (PM2.5 with environment) ===
    d["pm_temp"] = d["PM2.5"] * d["TEMP"]
    d["pm_temp_sq"] = (d["PM2.5"] * d["TEMP"]) ** 2
    d["pm_humidity"] = d["PM2.5"] * d["humidity"]
    d["pm_humidity_sq"] = (d["PM2.5"] * d["humidity"]) ** 2
    d["pm_gasValue"] = d["PM2.5"] * d["gasValue"]
    
    # === OTHER POLLUTANTS (PM10, gasValue multi-scale lags) ===
    for window, shift in [(7, 3), (15, 7), (30, 15)]:
        d[f"PM10_lag_w{window}"] = d["PM10"].rolling(window).mean().shift(shift)
        d[f"gasValue_lag_w{window}"] = d["gasValue"].rolling(window).mean().shift(shift)
    
    # === TEMPORAL FEATURES ===
    d["hour"] = d["timestamp"].dt.hour
    d["weekday"] = d["timestamp"].dt.weekday
    d["day"] = d["timestamp"].dt.day
    d["hour_sin"] = np.sin(2 * np.pi * d["hour"] / 24)
    d["hour_cos"] = np.cos(2 * np.pi * d["hour"] / 24)
    d["weekday_sin"] = np.sin(2 * np.pi * d["weekday"] / 7)
    d["weekday_cos"] = np.cos(2 * np.pi * d["weekday"] / 7)

    # === TARGETS (1-hour ahead from 2-minute feed = 30 rows ahead) ===
    d["TEMP_next"] = d["TEMP"].shift(-30)
    d["humidity_next"] = d["humidity"].shift(-30)
    d["PM2.5_next"] = d["PM2.5"].shift(-30)

    if "AQI" in d.columns:
        d["AQI_next"] = d["AQI"].shift(-30)
    if "CACI" in d.columns:
        d["CACI_next"] = d["CACI"].shift(-30)

    return d


def run_india_local_adaptation():
    india_raw = load_adaptation_raw()
    india_df = to_canonical_frame(india_raw)
    india_eng = build_engineered_from_india(india_df)

    # All enhanced features for general models (TEMP, HUM)
    base_features = ["PM2.5", "PM10", "gasValue", "TEMP", "humidity", "day", "hour_sin", "hour_cos", "weekday_sin", "weekday_cos"]
    pm25_specific = [col for col in india_eng.columns if any(x in col for x in ["PM2.5_lag", "PM2.5_std", "PM2.5_roc", "PM2.5_ema", "PM2.5_pct", "PM2.5_iqr", "pm_"])]
    lagfeatures_other = [col for col in india_eng.columns if any(x in col for x in ["PM10_lag", "gasValue_lag"])]
    
    deploy_features = base_features + pm25_specific + lagfeatures_other

    train_df = india_eng.dropna(subset=deploy_features + ["TEMP_next", "humidity_next", "PM2.5_next"]).copy()
    if len(train_df) < 500:
        raise ValueError("Not enough local adaptation rows. Collect more ThingSpeak history first.")

    print(f"Training data shape: {train_df.shape}, Feature count: {len(deploy_features)}")
    
    split_idx = int(len(train_df) * 0.8)
    X_train, X_test = train_df[deploy_features].iloc[:split_idx], train_df[deploy_features].iloc[split_idx:]

    # === TEMP & Humidity: RandomForest (stable) ===
    print("Training TEMP model...")
    temp_live = RandomForestRegressor(n_estimators=300, max_depth=15, random_state=42, n_jobs=-1)
    temp_live.fit(X_train, train_df["TEMP_next"].iloc[:split_idx])
    pred_temp = temp_live.predict(X_test)
    temp_mae = mean_absolute_error(train_df["TEMP_next"].iloc[split_idx:], pred_temp)
    
    print("Training HUM model...")
    hum_live = RandomForestRegressor(n_estimators=300, max_depth=15, random_state=42, n_jobs=-1)
    hum_live.fit(X_train, train_df["humidity_next"].iloc[:split_idx])
    pred_hum = hum_live.predict(X_test)
    hum_mae = mean_absolute_error(train_df["humidity_next"].iloc[split_idx:], pred_hum)

    # === PM2.5: Try XGBoost first (better for noisy data), fallback to RF ===
    print("Training PM2.5 model...")
    if HAS_XGBOOST:
        pm25_live = xgb.XGBRegressor(
            n_estimators=500,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.9,
            random_state=42,
            n_jobs=-1,
            objective='reg:squarederror'
        )
    else:
        print("  (XGBoost unavailable, using RandomForest with aggressive tuning)")
        pm25_live = RandomForestRegressor(n_estimators=800, max_depth=20, min_samples_split=2, random_state=42, n_jobs=-1)
    
    pm25_live.fit(X_train, train_df["PM2.5_next"].iloc[:split_idx])
    pred_pm = pm25_live.predict(X_test)
    pm25_mae = mean_absolute_error(train_df["PM2.5_next"].iloc[split_idx:], pred_pm)

    print(f"\n✅ India Model Accuracy:")
    print(f"  TEMP MAE : {round(temp_mae, 3)}")
    print(f"  HUM MAE  : {round(hum_mae, 3)}")
    print(f"  PM2.5 MAE: {round(pm25_mae, 3)} (improved from 33.009)")

    # AQI/CACI retrain only if true labels exist
    trained_aqi_caci = False
    if "AQI_next" in train_df.columns and "CACI_next" in train_df.columns:
        aqi_rows = train_df.dropna(subset=["AQI_next"]).copy()
        caci_rows = train_df.dropna(subset=["CACI_next"]).copy()
        if len(aqi_rows) > 500 and len(caci_rows) > 500:
            split_aqi = int(len(aqi_rows) * 0.8)
            split_caci = int(len(caci_rows) * 0.8)

            aqi_live = RandomForestRegressor(n_estimators=500, random_state=42, n_jobs=-1)
            caci_live = RandomForestRegressor(n_estimators=500, random_state=42, n_jobs=-1)

            aqi_live.fit(aqi_rows[deploy_features].iloc[:split_aqi], aqi_rows["AQI_next"].iloc[:split_aqi])
            caci_live.fit(caci_rows[deploy_features].iloc[:split_caci], caci_rows["CACI_next"].iloc[:split_caci])

            pred_aqi = aqi_live.predict(aqi_rows[deploy_features].iloc[split_aqi:])
            pred_caci = caci_live.predict(caci_rows[deploy_features].iloc[split_caci:])

            print("India AQI MAE:", round(mean_absolute_error(aqi_rows["AQI_next"].iloc[split_aqi:], pred_aqi), 3))
            print("India CACI MAE:", round(mean_absolute_error(caci_rows["CACI_next"].iloc[split_caci:], pred_caci), 3))

            joblib.dump(aqi_live, "aqi_model_live.pkl")
            joblib.dump(caci_live, "caci_model_live.pkl")
            trained_aqi_caci = True

    joblib.dump(temp_live, "temp_model_live.pkl")
    joblib.dump(hum_live, "humidity_model_live.pkl")
    joblib.dump(pm25_live, "pm25_model_live.pkl")

    if trained_aqi_caci:
        print("\n✅ Saved updated India-adapted live models (AQI/CACI/TEMP/HUM/PM2.5)")
    else:
        print("\n✅ Saved updated India-adapted TEMP/HUM/PM2.5 live models.")
        print("   AQI/CACI not retrained (local ground-truth labels unavailable).")


def preview_adaptation_data_head(n=5):
    sample = to_canonical_frame(load_adaptation_raw())
    print(sample.head(n))


# Run when your local export/API config is ready:
run_india_local_adaptation()

Training data shape: (1321, 44), Feature count: 37
Training TEMP model...
Training HUM model...
Training PM2.5 model...

✅ India Model Accuracy:
  TEMP MAE : 0.322
  HUM MAE  : 4.176
  PM2.5 MAE: 26.789 (improved from 33.009)

✅ Saved updated India-adapted TEMP/HUM/PM2.5 live models.
   AQI/CACI not retrained (local ground-truth labels unavailable).
